# 1. Montar Google Drive e Importaciones

In [ ]:
# ==========================================
# 1. MONTAR GOOGLE DRIVE E INSTALAR DEPENDENCIAS
# ==========================================
from google.colab import drive
import os
import json
import orjson
import pandas as pd
import gc
from collections import defaultdict


drive.mount('/content/drive')

Mounted at /content/drive


# 2. Configuración de Rutas

In [ ]:
# ==========================================
# 2. CONFIGURACIÓN DE RUTAS SEGÚN TUS CARPETAS
# ==========================================
BASE_DRIVE = '/content/drive/MyDrive/TFG/data'

DIR_PEMR = os.path.join(BASE_DRIVE, 'PatientEncounterMedRequest')
DIR_VARIOS = os.path.join(BASE_DRIVE, 'Varios')
DIR_REST = os.path.join(BASE_DRIVE, 'Rest')

# Carpeta local temporal en Colab para guardar el resultado de salida sin llenar tu PC
OUTPUT_DIR = '/content/output_tfg'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'medication_request_dataset_full.jsonl')

# Rutas de los archivos distribuidos en tus 3 carpetas
PATIENT_FILE = os.path.join(DIR_PEMR, 'MimicPatient.ndjson')
MED_REQUEST_FILE = os.path.join(DIR_PEMR, 'MimicMedicationRequest.ndjson')
ENCOUNTER_FILES = [
    os.path.join(DIR_PEMR, 'MimicEncounter.ndjson'),
    os.path.join(DIR_REST, 'MimicEncounterED.ndjson')
]
CONDITION_FILES = [
    os.path.join(DIR_VARIOS, 'MimicCondition.ndjson'),
    os.path.join(DIR_REST, 'MimicConditionED.ndjson')
]
MED_FILE = os.path.join(DIR_VARIOS, 'MimicMedication.ndjson')
ADMIN_FILE = os.path.join(DIR_REST, 'MimicMedicationAdministration.ndjson')
PROCEDURE_FILES = [
    os.path.join(DIR_VARIOS, 'MimicProcedure.ndjson')
]

# El CSV de prescripciones con los identificadores de médicos
CSV_MEDICOS = os.path.join(BASE_DRIVE, 'prescriptions.csv.gz')

# Ruta del CSV de pacientes con los slots temporales (para filtrar el sesgo de anonimización)
PATIENTS_CSV_FILE = os.path.join(BASE_DRIVE, 'patients.csv')

# 3. Herramientas Base (Iterador NDJSON y Función Segura *dig*)

In [ ]:
# ==========================================
# 3. HERRAMIENTAS BASE
# ==========================================

def iter_ndjson(path, stats):
    """
    Lee los archivos línea a línea de forma eficiente y los convierte de texto
    plano a diccionarios de Python usando orjson, incluyendo control de errores.
    """
    if not os.path.exists(path):
        print(f"⚠️ Aviso: No se encontró {path}")
        return
    with open(path, 'rt', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                yield orjson.loads(line)
            except (orjson.JSONDecodeError, ValueError):
                stats['json_errors'] += 1

def dig(d, *path, default=None):
    """
    Función de seguridad para trabajar con estándares FHIR.
    Evita KeyError cuando los datos anidados profundos no existen,
    devolviendo un valor por defecto de manera limpia.
    """
    for k in path:
        try:
            d = d[k]
        except (KeyError, IndexError, TypeError):
            return default
    return d

# 4. Funciones de Carga en RAM (Dimensiones y Mapeo Temporal)

In [ ]:
# ==========================================
# 3. FUNCIONES DE CARGA OPTIMIZADAS (FILTRADO EN ORIGEN)
# ==========================================
import gc

def cargar_pacientes_filtrados(filepath, mapa_slots, stats):
    """
    Carga únicamente los pacientes que pertenecen estrictamente al slot 2008-2010.
    """
    print("⏳ Cargando y filtrando pacientes (Slot 2008-2010)...")
    diccionario = {}
    pacientes_validos = set()

    for paciente in iter_ndjson(filepath, stats):
        fhir_uuid = paciente.get('id')

        # Extraer subject_id numérico
        subject_id_num = None
        for ident in paciente.get('identifier', []):
            if 'mimic/identifier/patient' in ident.get('system', ''):
                subject_id_num = str(ident.get('value'))
                break

        slot = mapa_slots.get(subject_id_num, 'Desconocido')

        # FILTRO INICIAL: Solo guardamos pacientes del slot 2008-2010
        if slot == '2008 - 2010':
            pacientes_validos.add(fhir_uuid)
            diccionario[fhir_uuid] = {
                'subject_id': subject_id_num,
                'anchor_year_group': slot,
                'patient_gender': paciente.get('gender', 'unknown'),
                'patient_birthdate': paciente.get('birthDate', 'unknown')
            }

    print(f"✅ Pacientes filtrados en RAM: {len(diccionario):,}")
    return diccionario, pacientes_validos


def cargar_encounters_filtrados(filepaths, pacientes_validos_ids, stats):
    """Carga solo los encuentros (ingresos) de nuestros pacientes válidos."""
    print("⏳ Cargando ingresos hospitalarios (filtrados por cohorte)...")
    diccionario = {}
    encuentros_validos = set()

    for filepath in filepaths:
        for enc in iter_ndjson(filepath, stats):
            pat_ref = dig(enc, 'subject', 'reference', default='')
            pat_id = pat_ref.rsplit('/', 1)[-1] if '/' in pat_ref else ''

            # Solo guardamos el encuentro si pertenece a un paciente de nuestra cohorte
            if pat_id in pacientes_validos_ids:
                enc_id = enc.get('id')
                encuentros_validos.add(enc_id)
                diccionario[enc_id] = {
                    'encounter_class': dig(enc, 'class', 'code', default='unknown'),
                    'encounter_start': dig(enc, 'period', 'start', default=''),
                    'encounter_end': dig(enc, 'period', 'end', default=''),
                    'encounter_service_type': dig(enc, 'serviceType', 'coding', 0, 'code', default='unknown')
                }
    print(f"✅ Encuentros filtrados cargados: {len(diccionario):,}")
    return diccionario, encuentros_validos


def cargar_diagnosticos_filtrados(filepaths, encuentros_validos_ids, stats):
    """Carga solo los diagnósticos de los encuentros de nuestra cohorte."""
    print("⏳ Cargando diagnósticos (filtrados)...")
    por_enc = defaultdict(dict)
    for filepath in filepaths:
        for cond in iter_ndjson(filepath, stats):
            enc_ref = dig(cond, 'encounter', 'reference', default='')
            enc_id = enc_ref.rsplit('/', 1)[-1] if '/' in enc_ref else ''

            if enc_id in encuentros_validos_ids:
                cod = dig(cond, 'code', 'coding', 0, 'code', default='UNKNOWN')
                txt = dig(cond, 'code', 'coding', 0, 'display', default='No description')
                por_enc[enc_id][cod] = txt
    return por_enc


def cargar_procedimientos_filtrados(filepaths, encuentros_validos_ids, stats):
    """Carga solo los procedimientos de los encuentros de nuestra cohorte."""
    print("⏳ Cargando procedimientos (filtrados)...")
    por_enc = defaultdict(dict)
    for filepath in filepaths:
        for proc in iter_ndjson(filepath, stats):
            enc_ref = dig(proc, 'encounter', 'reference', default='')
            if not enc_ref:
                enc_ref = proc.get('encounter', '')
                if isinstance(enc_ref, dict):
                    enc_ref = enc_ref.get('reference', '')
            enc_id = enc_ref.rsplit('/', 1)[-1] if '/' in enc_ref else str(enc_ref)

            if enc_id in encuentros_validos_ids:
                cod = dig(proc, 'code', 'coding', 0, 'code', default='UNKNOWN')
                txt = dig(proc, 'code', 'coding', 0, 'display', default='No description')
                por_enc[enc_id][cod] = txt
    return por_enc


def cargar_medicamentos(filepath, stats):
    """Carga el catálogo de medicamentos (este es pequeño y se queda entero)."""
    print("⏳ Cargando catálogo de medicamentos...")
    diccionario = {}
    for med in iter_ndjson(filepath, stats):
        med_id = med.get('id')
        med_name = "unknown"
        for ident in med.get('identifier', []):
            if 'mimic-medication-name' in ident.get('system', ''):
                med_name = ident.get('value', 'unknown')
                break
        if med_name == "unknown":
            med_name = dig(med, 'code', 'text', default="unknown")
        diccionario[med_id] = med_name
    return diccionario


def cargar_administraciones(filepath, stats):
    """Carga los registros de administración."""
    print("⏳ Cargando registros de administración...")
    administraciones = set()
    for admin in iter_ndjson(filepath, stats):
        req_ref = dig(admin, 'request', 'reference', default='')
        req_id = req_ref.rsplit('/', 1)[-1] if '/' in req_ref else ''
        if req_id:
            administraciones.add(req_id)
    return administraciones

# 5. Procesamiento Maestro (ETL y Filtrado del Slot 2008-2010)

In [ ]:
# ==========================================
# 5. PROCESAMIENTO MAESTRO (ETL)
# ==========================================
def procesar_etl_completo():
    print("⏳ Cargando el mapa de médicos...")
    if not os.path.exists(CSV_MEDICOS):
        print(f"❌ No se encuentra el archivo CSV en {CSV_MEDICOS}.")
        return

    df = pd.read_csv(CSV_MEDICOS, usecols=['pharmacy_id', 'order_provider_id'], dtype=str)
    mapa_medicos = pd.Series(df.order_provider_id.values, index=df.pharmacy_id.astype(str)).to_dict()
    print(f"✅ Mapa de médicos cargado: {len(mapa_medicos)} registros indexados.")

    # 1. Cargar el CSV de pacientes de los tutores y crear el mapa de slots
    if not os.path.exists(PATIENTS_CSV_FILE):
        print(f"❌ No se encuentra el archivo patients.csv en {PATIENTS_CSV_FILE}.")
        return

    df_pat = pd.read_csv(PATIENTS_CSV_FILE, dtype={'subject_id': str})
    mapa_slots = dict(zip(df_pat['subject_id'], df_pat['anchor_year_group']))
    print(f"✅ Mapa de slots cargado: {len(mapa_slots)} pacientes indexados.")

    if not os.path.exists(MED_REQUEST_FILE):
        print(f"❌ No se encuentra el archivo principal en: {MED_REQUEST_FILE}")
        return

    stats = defaultdict(int)

    # 2. Cargar dimensiones en RAM pasando el mapa de slots
    pacientes = cargar_pacientes(PATIENT_FILE, mapa_slots, stats)

    diagnosticos = cargar_diagnosticos(CONDITION_FILES, stats)
    procedimientos = cargar_procedimientos(PROCEDURE_FILES, stats)
    medicamentos = cargar_medicamentos(MED_FILE, stats)
    encounters = cargar_encounters(ENCOUNTER_FILES, stats)
    administraciones = cargar_administraciones(ADMIN_FILE, stats)

    gc.collect()

    print(f"🚀 Iniciando ETL completo con filtrado temporal (2008-2010)...")

    contador_exito = 0
    contador_total = 0

    with open(MED_REQUEST_FILE, 'rt', encoding='utf-8', errors='ignore') as f_in, \
         open(OUTPUT_FILE, 'wb') as f_out:

        for linea in f_in:
            linea_limpia = linea.strip()
            if not linea_limpia:
                continue
            try:
                receta = json.loads(linea_limpia)
            except (json.JSONDecodeError, MemoryError):
                stats['json_errors'] += 1
                continue

            contador_total += 1
            stats['total_recetas_procesadas'] += 1

            # 1. Obtención de IDs principales
            req_id = receta.get('id')
            patient_id = dig(receta, 'subject', 'reference', default='').rsplit('/', 1)[-1]
            encounter_id = dig(receta, 'encounter', 'reference', default='').rsplit('/', 1)[-1]

            # FILTRO ESTRICTO: Comprobar si el paciente pertenece al slot 2008 - 2010
            datos_paciente = pacientes.get(patient_id, {})
            slot_temporal = datos_paciente.get('anchor_year_group', 'Desconocido')

            if slot_temporal != '2008 - 2010':
                continue # Descartamos al vuelo cualquier receta fuera de nuestro slot de interés

            # 2. Cruce para obtener el médico prescriptor
            pharmacy_id = None
            for ident in receta.get('identifier', []):
                val = str(ident.get('value', ''))
                if val:
                    pharmacy_id = val
                    break

            requester_id = mapa_medicos.get(pharmacy_id, None)

            # 🛑 FILTRO CLAVE: Si la receta no tiene médico asociado, la descartamos
            if not requester_id:
                stats['recetas_sin_medico_descartadas'] += 1
                continue

            contador_exito += 1

            # 3. Resolución de nombres de medicamentos
            medication_id = dig(receta, 'medicationReference', 'reference', default='').rsplit('/', 1)[-1]
            medication_name = medicamentos.get(medication_id, 'unknown') if medication_id else 'unknown'

            if medication_name == 'unknown':
                concept_text = dig(receta, 'medicationCodeableConcept', 'text')
                if concept_text:
                    medication_name = concept_text
                else:
                    concept_display = dig(receta, 'medicationCodeableConcept', 'coding', 0, 'display')
                    concept_code = dig(receta, 'medicationCodeableConcept', 'coding', 0, 'code')
                    if concept_display: medication_name = concept_display
                    elif concept_code: medication_name = concept_code

            if medication_name == 'unknown':
                texto_dosis = dig(receta, 'dosageInstruction', 0, 'text', default='')
                route = dig(receta, 'dosageInstruction', 0, 'route', 'coding', 0, 'code', default='')
                if texto_dosis and (route == 'IV' or any(term in texto_dosis.lower() for term in ['bag', 'bottle', 'premix', 'soln', 'iv', 'infusion'])):
                    medication_name = f"IV Fluid / Mix: {texto_dosis}"
                else:
                    medication_name = f"Unknown Oral/Other: {texto_dosis}" if texto_dosis else "unknown"

            if patient_id not in pacientes: stats['miss_patient'] += 1
            if encounter_id not in encounters: stats['miss_encounter'] += 1
            if medication_name == 'unknown': stats['miss_medication'] += 1

            # Recuperación de datos del encuentro
            datos_encounter = encounters.get(encounter_id, {
                'encounter_class': 'unknown',
                'encounter_start': '',
                'encounter_end': '',
                'encounter_service_type': 'unknown'
            })

            dict_diags = diagnosticos.get(encounter_id, {})
            diags_texto = " | ".join([f"{cod}: {txt}" for cod, txt in dict_diags.items()]) if dict_diags else "No diagnosis recorded"

            dict_procs = procedimientos.get(encounter_id, {})
            procs_texto = " | ".join([f"{cod}: {txt}" for cod, txt in dict_procs.items()]) if dict_procs else "No procedure recorded"

            fue_administrado = req_id in administraciones

            dosage_text = dig(receta, 'dosageInstruction', 0, 'text', default='')
            route_code = dig(receta, 'dosageInstruction', 0, 'route', 'coding', 0, 'code', default='')
            timing_code = dig(receta, 'dosageInstruction', 0, 'timing', 'code', 'coding', 0, 'code', default='')
            dose_value = dig(receta, 'dosageInstruction', 0, 'doseAndRate', 0, 'doseQuantity', 'value')
            dose_unit = dig(receta, 'dosageInstruction', 0, 'doseAndRate', 0, 'doseQuantity', 'unit', default='')

            fila_aplanada = {
                'IdMedicationRequest': req_id,
                'authoredOn': receta.get('authoredOn'),
                'pharmacy_id': pharmacy_id,
                'order_provider_id': requester_id,
                'fue_administrado': fue_administrado,

                'encounter_id': encounter_id,
                'encounter_class': datos_encounter['encounter_class'],
                'encounter_start': datos_encounter['encounter_start'],
                'encounter_end': datos_encounter['encounter_end'],
                'encounter_service_type': datos_encounter['encounter_service_type'],
                'encounter_diagnoses': diags_texto,
                'encounter_procedures': procs_texto,

                'patient_id': patient_id,
                'subject_id': datos_paciente.get('subject_id'),
                'anchor_year_group': slot_temporal,
                'patient_gender': datos_paciente.get('patient_gender', 'unknown'),
                'patient_birthdate': datos_paciente.get('patient_birthdate', 'unknown'),

                'medication_id': medication_id,
                'medication_name': medication_name,
                'route_code': route_code,
                'timing_code': timing_code,
                'dose_value': dose_value,
                'dose_unit': dose_unit,
                'dosage_text': dosage_text
            }
            f_out.write(orjson.dumps(fila_aplanada) + b'\n')

            if contador_total % 100000 == 0:
                print(f"   Procesadas {contador_total} recetas...")

    print(f"\n🎉 ¡ETL completado con éxito en Google Colab!")
    print(f"   - Total de recetas analizadas (slot 2008-2010): {contador_exito}")
    print(f"   - Archivo final guardado temporalmente en: {OUTPUT_FILE}")

    # Copiar el resultado final directamente de vuelta a tu Google Drive
    import shutil
    drive_output_path = os.path.join(BASE_DRIVE, 'medication_request_dataset_full.jsonl')
    shutil.copy(OUTPUT_FILE, drive_output_path)
    print(f"📁 Archivo final copiado automáticamente a tu Google Drive en: {drive_output_path}")

if __name__ == "__main__":
    procesar_etl_completo()

⏳ Cargando el mapa de médicos...
✅ Mapa de médicos cargado: 17779025 registros indexados.
✅ Mapa de slots cargado: 364627 pacientes indexados.
⏳ Cargando pacientes y vinculando slots temporales...
⏳ Cargando diagnósticos...
⏳ Cargando procedimientos...
⏳ Cargando catálogo de medicamentos...
⏳ Cargando ingresos hospitalarios...
⏳ Cargando registros de administración...
🚀 Iniciando ETL completo con filtrado temporal (2008-2010)...
   Procesadas 100000 recetas...
   Procesadas 500000 recetas...
   Procesadas 1100000 recetas...
   Procesadas 1900000 recetas...
   Procesadas 2000000 recetas...
   Procesadas 3500000 recetas...
   Procesadas 3600000 recetas...
   Procesadas 4000000 recetas...
   Procesadas 7600000 recetas...
   Procesadas 8300000 recetas...
   Procesadas 10900000 recetas...
   Procesadas 14800000 recetas...

🎉 ¡ETL completado con éxito en Google Colab!
   - Total de recetas analizadas (slot 2008-2010): 1056012
   - Archivo final guardado temporalmente en: /content/output_tfg/

# 6. Inspección de Resultados

SIGUE SIENDO EL MISMO patient_id

In [ ]:
# ==========================================
# 6. INSPECCIÓN DE RESULTADOS
# ==========================================

import json
import os
import random

# Ruta de tu archivo resultante en Google Drive
archivo_resultado = '/content/drive/MyDrive/TFG/data/medication_request_dataset_full.jsonl'

if not os.path.exists(archivo_resultado):
    print(f"❌ No se encuentra el archivo en: {archivo_resultado}")
else:
    print(f"🔍 Inspeccionando registros aleatorios del dataset filtrado (Slot 2008-2010):\n")
    print("-" * 60)

    lineas_cargadas = []
    with open(archivo_resultado, 'rt', encoding='utf-8') as f:
        # Leemos las primeras 1,000 líneas (o menos si el archivo es más corto)
        for i in range(1000):
            linea = f.readline()
            if not linea:
                break
            lineas_cargadas.append((i + 1, linea.strip()))

    if not lineas_cargadas:
        print("⚠️ El archivo de resultados está vacío.")
    else:
        # Seleccionamos hasta 20 al azar
        muestra_aleatoria = random.sample(lineas_cargadas, min(20, len(lineas_cargadas)))

        # Ordenamos por su número de línea original para que se muestren en orden de aparición
        muestra_aleatoria.sort(key=lambda x: x[0])

        for idx, (num_linea, contenido_linea) in enumerate(muestra_aleatoria):
            try:
                datos = json.loads(contenido_linea)
                print(f"Muestra {idx + 1} (Línea original #{num_linea}):")
                print(json.dumps(datos, indent=4, ensure_ascii=False))
            except json.JSONDecodeError:
                print(f"Muestra {idx + 1} (Línea #{num_linea} - Error de decodificación JSON):")
                print(contenido_linea)
            print("-" * 60)

🔍 Inspeccionando registros aleatorios del dataset filtrado (Slot 2008-2010):

------------------------------------------------------------
Muestra 1 (Línea original #79):
{
    "IdMedicationRequest": "fa8f645d-171a-51c3-9d61-5a4bf2e8337e",
    "authoredOn": "2157-12-07T14:51:02-05:00",
    "pharmacy_id": "18079690",
    "order_provider_id": "P24O72",
    "fue_administrado": false,
    "encounter_id": "562b529c-3e25-53e5-b077-2e050c5c49f0",
    "encounter_class": "EMER",
    "encounter_start": "2157-12-07T14:47:00-05:00",
    "encounter_end": "2157-12-10T18:10:00-05:00",
    "encounter_service_type": "NMED",
    "encounter_diagnoses": "71590: Osteoarthrosis, unspecified whether generalized or localized, site unspecified | 4019: Unspecified essential hypertension | 49390: Asthma, unspecified type, unspecified | 2449: Unspecified acquired hypothyroidism | 2724: Other and unspecified hyperlipidemia | 27801: Morbid obesity | 135: Sarcoidosis | 32723: Obstructive sleep apnea (adult)(pediatri